## Figure 5. Validation of the NMWI across independent datasets and disease phenotypes. 
Top panels display NMWI score distributions, and bottom panels show Shannon diversity in the same validation cohort for comparison. **(a)** Pooled healthy vs. non-healthy samples, demonstrating significant separation in NMWI scores (Mann–Whitney U test, P = 2.31 × 10–18; |Cliff’s δ| = 0.58), whereas Shannon diversity did not differ significantly (P = 0.058, |δ| = 0.13). **(b)** Distributions across independent healthy cohorts. Asterisks indicate statistically significant differences relative to pooled non-healthy samples (Mann–Whitney U test, P < 0.01); |Cliff's δ| is shown above each cohort. **(c)** Study-specific distributions across non-healthy cohorts. Asterisks denote significant differences relative to pooled healthy samples (Mann–Whitney U test, P < 0.01); |Cliff's δ| is shown above each cohort .


In [2]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import glob, warnings, sys, runpy
from scipy.stats import entropy, mannwhitneyu
from sklearn.metrics import balanced_accuracy_score
from matplotlib.patches import Patch
import matplotlib
# matplotlib.use("cairo")
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API")

project_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "NMWI")
os.chdir(project_root)

In [1]:
%matplotlib inline

# METADATA LOADING
vars_dict = runpy.run_path("Codes/Auxiliary_files/Generate_data_index.py") 
globals().update(vars_dict)
X_new = X.copy()

%run -i Codes/Auxiliary_files/NMWI.py # Run NMWI model

NameError: name 'runpy' is not defined

- X: DataFrame (rows=samples, cols=taxonomy), index includes bioproject/sample_id RELATIVE ABUNDANCE
- y: Series/array-like of True/False (same index as X), True (1) =control, False=case

### Processing data for the validation set

In [ ]:
## Working dataframe for validation:
# Load metadata
df_validation = pd.read_excel(file_path, sheet_name='Table S1', skiprows=1)
df_validation = df_validation[df_validation["Set"] == "Validation"] # Unique studies = 11
df_validation["Citation"] = df_validation["Author"].astype(str) + " et al., " + df_validation["Publication Year"].astype(str)

sra_data_validation = pd.read_excel(file_path, sheet_name='Table S2', skiprows=1)
sra_data_validation = sra_data_validation[sra_data_validation["Set"]=="Validation"] # samples 1654
sra_data_validation = sra_data_validation[['Sample ID', 'BioProject Number', 'Category', 'Disease']]


def read_genus_table(path):
    with open(path, 'r', encoding='utf-8') as f:
        lines_validation = f.readlines()
    header_idx_validation = next((i for i, line in enumerate(lines_validation) if line.lstrip().startswith('#OTU ID')), None)
    header_validation = lines_validation[header_idx_validation].lstrip('#').strip().split('\t')
    table_data_validation = pd.read_csv(path, sep='\t', header=None, names=header_validation, skiprows=header_idx_validation + 1, dtype=str)
    sample_cols_validation = [c for c in table_data_validation.columns if c != 'OTU ID']
    table_data_validation[sample_cols_validation] = table_data_validation[sample_cols_validation].apply(pd.to_numeric, errors='coerce')
    return table_data_validation

paths_validation = sorted(glob.glob(os.path.join('Feature_tables', '**', '*_final_table_depth.tsv'), recursive=True))
allowed_projects_validation = set(df_validation["BioProject Number"].dropna().astype(str))
filtered_files_validation = [f for f in paths_validation if os.path.basename(f).split("_")[2] in allowed_projects_validation]
print(f"Found {len(filtered_files_validation)} genus tables in validation")

# Merge all genus tables
merged_validation = None
for p in filtered_files_validation:
    table_data_validation = read_genus_table(p)
    sample_cols_validation = [c for c in table_data_validation.columns if c != 'OTU ID']
    if merged_validation is None:
        merged_validation = table_data_validation.copy()
    else:
        merged_sample_cols_validation = [c for c in merged_validation.columns if c != 'OTU ID']
        dupes_validation = set(sample_cols_validation) & set(merged_sample_cols_validation)
        if dupes_validation:
            raise ValueError(f"Duplicate sample IDs found: {dupes_validation} in {p}")
        merged_validation = merged_validation.merge(table_data_validation, on='OTU ID', how='outer')

sample_cols_validation = [c for c in merged_validation.columns if c != 'OTU ID']
merged_validation[sample_cols_validation] = merged_validation[sample_cols_validation].apply(pd.to_numeric, errors='coerce').fillna(0.0)
merged_validation = merged_validation[['OTU ID'] + sorted(sample_cols_validation)]
merged_validation["OTU ID"] = merged_validation["OTU ID"].apply(simplify_taxonomy)
merged_validation = merged_validation.groupby("OTU ID", as_index=False).sum(numeric_only=True)

ra_validation = merged_validation.set_index('OTU ID').copy()
filtered_t_val = ra_validation.T.reset_index().rename(columns={'index': 'Sample ID'})

# Merge metadata
validation_meta = filtered_t_val.merge(sra_data_validation, on='Sample ID', how='left')
validation_meta = validation_meta.merge(df_validation, on='BioProject Number', how='left')
validation_meta = validation_meta[validation_meta["Sample ID"].isin(sra_data_validation["Sample ID"])].set_index(['BioProject Number', 'Sample ID'])
print(f"Number of unique studies: {validation_meta.index.get_level_values(0).nunique()}")

X_val_transform = validation_meta[[c for c in validation_meta.columns if c.startswith("d__")]] # 350 rows x 1,627 cols

# Align validation features to training features (184) and normalize the samples
train_features = list(NMWI.feature_names_in_)
missing = set(train_features) - set(X_val_transform.columns)
for f in missing:
    X_val_transform[f] = 0

# Validation set; correct labels
X_val_raw = X_val_transform[train_features].copy()

X_aligned = X_val_raw.div(X_val_raw.sum(axis=1), axis=0)
print(f"\nCheck that samples have been renormalized: {min(X_aligned.sum(axis=1)):.3f}")
y_val = validation_meta["Category"].map({'case': False, 'control': True})

# FINAL WORKING TABLE
validation_set = X_aligned.reset_index().merge(
    validation_meta.reset_index()[["Sample ID", "Category", "Disease", "Citation"]],
    how="right", on="Sample ID"
)
print(f"Dimensions of the validation data set:")
print(f"Number of samples: {len(validation_set)}; number of taxa: {validation_set[[c for c in validation_set.columns if c.startswith("d__")]].shape[1]}")

# Count samples per category
samples_per_category = validation_set.groupby("Category")["Sample ID"].nunique().reset_index(name="n_sample_id").sort_values("n_sample_id", ascending=False)
print("\nSample ID per Category:")
print(samples_per_category.to_string(index=False))

# Validation decision scores
NMWI_scores = pd.Series(NMWI.decision_function(X_aligned), index=X_aligned.index, name="NMWI")
globals().update(vars_dict)


In [ ]:
# Validation decision scores
NMWI_scores = pd.Series(
    NMWI.decision_function(X_aligned),
    index=X_aligned.index,
    name="NMWI"
)

df = validation_set.reset_index()[["BioProject Number","Sample ID","Category","Disease"]].copy()
df["NMWI"] = NMWI_scores.values
df["y"] = df["Category"].map({"case":False,"control":True})
df["pred"] = df["NMWI"] > 0
df["Disease_clean"] = df["Disease"].astype(str).str.strip().str.upper().str.replace("-","",regex=False)

acute = {"RTI","COVID19"}

overall = balanced_accuracy_score(df["y"],df["pred"])
acute_df = df[(df.Category=="control") | ((df.Category=="case") & df.Disease_clean.isin(acute))]
nonacute_df = df[(df.Category=="control") | ((df.Category=="case") & ~df.Disease_clean.isin(acute))]

print(f"Overall BA:   {overall:.4f}")
print(f"Acute BA:     {balanced_accuracy_score(acute_df.y,acute_df.pred):.5f} (n={len(acute_df)})")
print(f"Non-acute BA: {balanced_accuracy_score(nonacute_df.y,nonacute_df.pred):.5f} (n={len(nonacute_df)})")

### Figure 5a-c; Top


In [ ]:
# Prepare metadata
cols = ["Category", "Disease", "BioProject Number", "Citation"]
meta_val = validation_set.reset_index()[cols].copy()
meta_val["NMWI"] = NMWI_scores.to_numpy().ravel()
meta_val[cols] = meta_val[cols].astype(str)
meta_val["Study_Label"] = meta_val["Citation"] + "\n(" + meta_val["BioProject Number"] + ")"

cases = meta_val.query("Category == 'case'").copy()
controls = meta_val.query("Category == 'control'").copy()

def study_order(df):
    label_map = df.drop_duplicates("BioProject Number").set_index("BioProject Number")["Study_Label"]
    return df.groupby("BioProject Number")["NMWI"].median().sort_values(ascending=False).index.map(label_map).tolist()

case_order = study_order(cases)
control_order = study_order(controls)

# Plot
ylim = 6
fig, axes = plt.subplots(1, 3, figsize=(18, 8), dpi=300, sharey=True,
                         gridspec_kw={"width_ratios": [1.2, 4, 2.7]})

def violin_box(ax, data, x, order, color, title, width=1.1):
    sns.violinplot(data=data, x=x, y="NMWI", order=order, color=color, inner=None,
                   width=width, cut=0, ax=ax)
    sns.boxplot(data=data, x=x, y="NMWI", order=order, width=0.3, showcaps=True,
                boxprops={"facecolor": "none", "zorder": 3}, showfliers=False,
                whiskerprops={"linewidth": 0.5}, ax=ax)
    ax.set(title=title, xlabel="", ylabel="NMWI Score", ylim=(-ylim, ylim))
    ax.axhline(0, color="black", linestyle="--", linewidth=1)

violin_box(axes[0], meta_val, "Category", ["control", "case"], "lightgray", "NMWI Validation")
violin_box(axes[1], cases, "Study_Label", case_order, "lightcoral", "Cases")
violin_box(axes[2], controls, "Study_Label", control_order, "lightblue", "Controls")

for ax in axes[1:]:
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
# plt.savefig("Figures/Fig5a.pdf")
plt.show()

# Case vs control comparison
sub = meta_val[["Category", "NMWI"]].dropna()
case_vals = sub.loc[sub["Category"] == "case", "NMWI"]
control_vals = sub.loc[sub["Category"] == "control", "NMWI"]

u, pval = mannwhitneyu(case_vals, control_vals, alternative="two-sided")
cliffs_delta = 2 * u / (len(case_vals) * len(control_vals)) - 1

results = pd.DataFrame([{
    "Group_1": "case", "Group_2": "control",
    "N_1": len(case_vals), "N_2": len(control_vals),
    "Cliffs_delta": cliffs_delta, "MannWhitney_p": pval
}])

results

In [ ]:
from scipy.stats import mannwhitneyu

healthy_vals = meta_val.loc[meta_val["Category"] == "control", "NMWI"].dropna()
results = []

for project, df in meta_val.query("Category == 'case'").groupby("BioProject Number"):
    case_vals = df["NMWI"].dropna()
    if case_vals.empty or healthy_vals.empty:
        continue

    u, pval = mannwhitneyu(case_vals, healthy_vals, alternative="two-sided")
    results.append({
        "BioProject Number": project,
        "Citation": df["Citation"].iloc[0],
        "N_case": len(case_vals),
        "N_healthy": len(healthy_vals),
        "Cliffs_delta": 2 * u / (len(case_vals) * len(healthy_vals)) - 1,
        "MannWhitney_p": pval,
        "Case_median_NMWI": case_vals.median(),
        "Healthy_median_NMWI": healthy_vals.median()
    })

bioproject_vs_healthy = pd.DataFrame(results).sort_values("MannWhitney_p")
bioproject_vs_healthy

In [ ]:
# Pooled case samples
case_vals = meta_val.loc[meta_val["Category"] == "case", "NMWI"].dropna()
results = []

for project, df in meta_val.query("Category == 'control'").groupby("BioProject Number"):
    healthy_vals = df["NMWI"].dropna()
    if healthy_vals.empty or case_vals.empty:
        continue

    u, pval = mannwhitneyu(healthy_vals, case_vals, alternative="two-sided")
    results.append({
        "BioProject Number": project,
        "Citation": df["Citation"].iloc[0],
        "N_healthy": len(healthy_vals),
        "N_case": len(case_vals),
        "Cliffs_delta": 2 * u / (len(healthy_vals) * len(case_vals)) - 1,
        "MannWhitney_p": pval,
        "Healthy_median_NMWI": healthy_vals.median(),
        "Case_median_NMWI": case_vals.median()
    })

healthy_bioproject_vs_cases = pd.DataFrame(results).sort_values("MannWhitney_p")
healthy_bioproject_vs_cases

### Figure 5a-c. Diversity index; Bottom

In [ ]:
# Alpha diversity
filtered = pd.DataFrame({
    "Shannon": X_val_transform.T.apply(lambda x: entropy(x[x > 0]), axis=0),
    "Simpson": X_val_transform.T.apply(lambda x: 1 - ((x / x.sum()) ** 2).sum(), axis=0),
    "Chao1": X_val_transform.T.apply(
        lambda x: (x > 0).sum() + x.eq(1).sum() * (x.eq(1).sum() - 1) / (2 * (x.eq(2).sum() + 1)),
        axis=0)
})

filtered.index.name = "Sample ID"
meta_diversity = filtered.merge(sra_data_validation, on="Sample ID", how="left")
filtered = meta_diversity.copy()

meta_cols = ["Category", "Disease", "BioProject Number"]
metrics = ["Shannon", "Simpson", "Chao1"]
meta_diversity[meta_cols] = meta_diversity[meta_cols].astype(str)
meta_diversity = meta_diversity.merge(df_validation[['Citation', 'BioProject Number']], on ='BioProject Number', how = 'left')


# Plotting data: pooled controls vs individual case diseases
cases = meta_diversity.query("Category == 'case'")
controls = meta_diversity.query("Category == 'control'")

plot_df_disease = pd.concat([
    controls.assign(Group="All controls"),
    cases.assign(Group=cases["Disease"])
], ignore_index=True)[["Group"] + metrics]

# Case vs control statistics
results = []

for metric in metrics:
    sub = meta_diversity[["Category", metric]].dropna()
    case_vals = sub.loc[sub["Category"] == "case", metric]
    control_vals = sub.loc[sub["Category"] == "control", metric]

    u, pval = mannwhitneyu(case_vals, control_vals, alternative="two-sided")
    delta = 2 * u / (len(case_vals) * len(control_vals)) - 1

    results.append({
        "Metric": metric,
        "Group_1": "case",
        "Group_2": "control",
        "N_1": len(case_vals),
        "N_2": len(control_vals),
        "Cliffs_delta": delta,
        "MannWhitney_p": pval
    })

alpha_diversity_results = pd.DataFrame(results)
print(alpha_diversity_results)

In [ ]:
metrics = ["Shannon", "Simpson", "Chao1"]
results = []

for metric in metrics:
    healthy_vals = meta_diversity.loc[meta_diversity["Category"] == "control", metric].dropna()

    for project, df in meta_diversity.query("Category == 'case'").groupby("BioProject Number"):
        case_vals = df[metric].dropna()
        if case_vals.empty or healthy_vals.empty:
            continue

        u, pval = mannwhitneyu(case_vals, healthy_vals, alternative="two-sided")

        results.append({
            "Metric": metric,
            "BioProject Number": project,
            "Disease": df["Disease"].iloc[0],
            "Citation": df["Citation"].iloc[0],
            "N_case": len(case_vals),
            "N_healthy": len(healthy_vals),
            "Cliffs_delta": 2 * u / (len(case_vals) * len(healthy_vals)) - 1,
            "MannWhitney_p": pval,
            "Significance": "significant" if pval < 0.01 else "ns"
        })

bioproject_vs_healthy = (
    pd.DataFrame(results)
    .sort_values(["Metric", "MannWhitney_p"])
)

bioproject_vs_healthy

In [ ]:
metric, BLUE, RED = "Shannon", "#2CA9DF", "#F15B5A"

df = meta_diversity.copy()
if "Citation" not in df:
    df = df.merge(meta_val[["BioProject Number","Citation"]].drop_duplicates(), on="BioProject Number", how="left")

df = df[df["Category"].isin(["control","case"])].dropna(subset=[metric])
controls, cases = df.query("Category == 'control'"), df.query("Category == 'case'")
control_vals, case_vals = controls[metric], cases[metric]

def mw(a, b):
    a, b = a.dropna(), b.dropna()
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    return 2*u/(len(a)*len(b)) - 1, p

def violin_box(ax, data, x, order, color=None, palette=None):
    sns.violinplot(data=data, x=x, y=metric, order=order, color=color, palette=palette,
                   hue=x if palette else None, legend=False, inner=None, cut=0, linewidth=.8, ax=ax)
    sns.boxplot(data=data, x=x, y=metric, order=order, width=.25, showfliers=False, ax=ax,
                boxprops={"facecolor":"white","edgecolor":"black","linewidth":1,"zorder":3},
                medianprops={"color":"black","linewidth":2.5,"zorder":5},
                whiskerprops={"color":"black","linewidth":1},
                capprops={"color":"black","linewidth":1})

def sig(p): return "*" if p < 0.01 else "ns"

def annotate(ax, data, x, plot_order, stats, key, ymax):
    stat_map = stats.set_index(key)
    for group in stat_map.index:
        i, r = plot_order.index(group), stat_map.loc[group]
        y = data.loc[data[x] == group, metric].max() + .025*ymax
        ax.text(i, y, f"δ={r['delta']:.2f}\n{sig(r['p'])}",
                ha="center", va="bottom", fontsize=10)

# Statistics
delta_all, p_all = mw(case_vals, control_vals)

disease_order = cases.query("Disease != 'nan'").groupby("Disease")[metric].median().sort_values(ascending=False).index.tolist()
disease_stats = []
for d in disease_order:
    vals = cases.loc[cases["Disease"] == d, metric]
    delta, p = mw(vals, control_vals)
    disease_stats.append({"Disease":d, "Disease_median":vals.median(),
                          "Healthy_median":control_vals.median(), "delta":delta, "p":p})
disease_stats = pd.DataFrame(disease_stats)

case_stats = []
for project, g in cases.groupby("BioProject Number"):
    delta, p = mw(g[metric], control_vals)
    case_stats.append({"BioProject Number":project, "Citation":g["Citation"].iloc[0],
                       "Case_median":g[metric].median(), "Healthy_median":control_vals.median(),
                       "delta":delta, "p":p})
case_stats = pd.DataFrame(case_stats)
case_order = case_stats.sort_values("Case_median", ascending=False)["BioProject Number"].tolist()

control_stats = []
for project, g in controls.groupby("BioProject Number"):
    delta, p = mw(g[metric], case_vals)
    control_stats.append({"BioProject Number":project, "Citation":g["Citation"].iloc[0],
                          "Healthy_median":g[metric].median(), "Case_median":case_vals.median(),
                          "delta":delta, "p":p})
control_stats = pd.DataFrame(control_stats)
control_order = control_stats.sort_values("Healthy_median", ascending=False)["BioProject Number"].tolist()

# Plot
sns.set_theme(style="whitegrid", context="paper")
fig, axes = plt.subplots(2, 2, figsize=(16,15), dpi=300)
axes = axes.ravel()

# A
order = ["control","case"]
violin_box(axes[0], df, "Category", order, palette={"control":BLUE,"case":RED})
ymax = df[metric].max()
axes[0].set_ylim(0, ymax*1.18)
axes[0].text(.5, ymax*1.07, f"δ={delta_all:.3f}\nP={p_all:.3g}",
             ha="center", fontsize=12, fontweight="bold")
axes[0].set_xticklabels(["Healthy","Non-healthy"])
axes[0].set(title="Shannon validation", xlabel="", ylabel="Shannon index")

# B
disease_df = pd.concat([
    controls.assign(Group="All controls"),
    cases[cases["Disease"].isin(disease_order)].assign(Group=lambda x:x["Disease"])
], ignore_index=True)

disease_plot_order = ["All controls"] + disease_order
violin_box(axes[1], disease_df, "Group", disease_plot_order,
           palette={"All controls":BLUE, **{d:RED for d in disease_order}})
ymax = disease_df[metric].max()
axes[1].set_ylim(0, ymax*1.18)
annotate(axes[1], disease_df, "Group", disease_plot_order,
         disease_stats.rename(columns={"Disease":"Group"}), "Group", ymax)
axes[1].set(title="Shannon by disease", xlabel="", ylabel="Shannon index")
axes[1].tick_params(axis="x", rotation=45)

# C
violin_box(axes[2], cases, "BioProject Number", case_order, color=RED)
ymax = cases[metric].max()
axes[2].set_ylim(0, ymax*1.18)
annotate(axes[2], cases, "BioProject Number", case_order, case_stats, "BioProject Number", ymax)
citation_map = case_stats.set_index("BioProject Number")["Citation"]
axes[2].set_xticklabels([citation_map[x] for x in case_order], rotation=45, ha="right")
axes[2].set(title="Shannon in Non-Healthy studies", xlabel="", ylabel="Shannon index")

# D
violin_box(axes[3], controls, "BioProject Number", control_order, color=BLUE)
ymax = controls[metric].max()
axes[3].set_ylim(0, ymax*1.18)
annotate(axes[3], controls, "BioProject Number", control_order, control_stats, "BioProject Number", ymax)
citation_map = control_stats.set_index("BioProject Number")["Citation"]
axes[3].set_xticklabels([citation_map[x] for x in control_order], rotation=45, ha="right")
axes[3].set(title="Shannon in Healthy studies", xlabel="", ylabel="Shannon index")

for label, ax in zip("abcd", axes):
    ax.text(-.08, 1.04, label, transform=ax.transAxes, fontsize=14, fontweight="bold")
    ax.tick_params(axis="both", labelsize=12)
    ax.xaxis.label.set_size(14)
    ax.yaxis.label.set_size(14)
    ax.title.set_size(14)
    ax.grid(axis="x", visible=False)
    ax.grid(axis="y", linestyle=":", linewidth=.6, alpha=.5)
    sns.despine(ax=ax)

fig.legend(handles=[Patch(facecolor=RED, label="Non-healthy"),
                    Patch(facecolor=BLUE, label="Healthy")],
           loc="lower center", ncol=2, frameon=False)

plt.tight_layout(rect=[0,.05,1,1])
plt.show()

---
Supplementary plots

In [ ]:
metric, BLUE, RED = "Chao1", "#2CA9DF", "#F15B5A"

df = meta_diversity.copy()
if "Citation" not in df:
    df = df.merge(meta_val[["BioProject Number","Citation"]].drop_duplicates(), on="BioProject Number", how="left")

df = df[df["Category"].isin(["control","case"])].dropna(subset=[metric])
controls, cases = df.query("Category == 'control'"), df.query("Category == 'case'")
control_vals, case_vals = controls[metric], cases[metric]


# Statistics
delta_all, p_all = mw(case_vals, control_vals)

disease_order = cases.query("Disease != 'nan'").groupby("Disease")[metric].median().sort_values(ascending=False).index.tolist()
disease_stats = []
for d in disease_order:
    vals = cases.loc[cases["Disease"] == d, metric]
    delta, p = mw(vals, control_vals)
    disease_stats.append({"Disease":d, "Disease_median":vals.median(),
                          "Healthy_median":control_vals.median(), "delta":delta, "p":p})
disease_stats = pd.DataFrame(disease_stats)

case_stats = []
for project, g in cases.groupby("BioProject Number"):
    delta, p = mw(g[metric], control_vals)
    case_stats.append({"BioProject Number":project, "Citation":g["Citation"].iloc[0],
                       "Case_median":g[metric].median(), "Healthy_median":control_vals.median(),
                       "delta":delta, "p":p})
case_stats = pd.DataFrame(case_stats)
case_order = case_stats.sort_values("Case_median", ascending=False)["BioProject Number"].tolist()

control_stats = []
for project, g in controls.groupby("BioProject Number"):
    delta, p = mw(g[metric], case_vals)
    control_stats.append({"BioProject Number":project, "Citation":g["Citation"].iloc[0],
                          "Healthy_median":g[metric].median(), "Case_median":case_vals.median(),
                          "delta":delta, "p":p})
control_stats = pd.DataFrame(control_stats)
control_order = control_stats.sort_values("Healthy_median", ascending=False)["BioProject Number"].tolist()

# Plot
sns.set_theme(style="whitegrid", context="paper")
fig, axes = plt.subplots(2, 2, figsize=(16,15), dpi=300)
axes = axes.ravel()

# A
order = ["control","case"]
violin_box(axes[0], df, "Category", order, palette={"control":BLUE,"case":RED})
ymax = df[metric].max()
axes[0].set_ylim(0, ymax*1.18)
axes[0].text(.5, ymax*1.07, f"δ={delta_all:.3f}\nP={p_all:.3g}",
             ha="center", fontsize=12, fontweight="bold")
axes[0].set_xticklabels(["Healthy","Non-healthy"])
axes[0].set(title="Chao1 validation", xlabel="", ylabel="Chao1 index")

# B
disease_df = pd.concat([
    controls.assign(Group="All controls"),
    cases[cases["Disease"].isin(disease_order)].assign(Group=lambda x:x["Disease"])
], ignore_index=True)

disease_plot_order = ["All controls"] + disease_order
violin_box(axes[1], disease_df, "Group", disease_plot_order,
           palette={"All controls":BLUE, **{d:RED for d in disease_order}})
ymax = disease_df[metric].max()
axes[1].set_ylim(0, ymax*1.18)
annotate(axes[1], disease_df, "Group", disease_plot_order,
         disease_stats.rename(columns={"Disease":"Group"}), "Group", ymax)
axes[1].set(title="Chao1 by disease", xlabel="", ylabel="Chao1 index")
axes[1].tick_params(axis="x", rotation=45)

# C
violin_box(axes[2], cases, "BioProject Number", case_order, color=RED)
ymax = cases[metric].max()
axes[2].set_ylim(0, ymax*1.18)
annotate(axes[2], cases, "BioProject Number", case_order, case_stats, "BioProject Number", ymax)
citation_map = case_stats.set_index("BioProject Number")["Citation"]
axes[2].set_xticklabels([citation_map[x] for x in case_order], rotation=45, ha="right")
axes[2].set(title="Chao1 in Non-Healthy studies", xlabel="", ylabel="Chao1 index")

# D
violin_box(axes[3], controls, "BioProject Number", control_order, color=BLUE)
ymax = controls[metric].max()
axes[3].set_ylim(0, ymax*1.18)
annotate(axes[3], controls, "BioProject Number", control_order, control_stats, "BioProject Number", ymax)
citation_map = control_stats.set_index("BioProject Number")["Citation"]
axes[3].set_xticklabels([citation_map[x] for x in control_order], rotation=45, ha="right")
axes[3].set(title="Chao1 in Healthy studies", xlabel="", ylabel="Chao1 index")

for label, ax in zip("abcd", axes):
    ax.text(-.08, 1.04, label, transform=ax.transAxes, fontsize=14, fontweight="bold")
    ax.tick_params(axis="both", labelsize=12)
    ax.xaxis.label.set_size(14)
    ax.yaxis.label.set_size(14)
    ax.title.set_size(14)
    ax.grid(axis="x", visible=False)
    ax.grid(axis="y", linestyle=":", linewidth=.6, alpha=.5)
    sns.despine(ax=ax)

fig.legend(handles=[Patch(facecolor=RED, label="Non-healthy"),
                    Patch(facecolor=BLUE, label="Healthy")],
           loc="lower center", ncol=2, frameon=False)

plt.tight_layout(rect=[0,.05,1,1])
plt.show()

In [ ]:
# pooled case samples
case_vals_pooled = (filtered.loc[filtered["Category"] == "case", "Shannon"].dropna())

results = []

healthy_projects = (filtered.loc[filtered["Category"] == "control", "BioProject Number"].dropna().astype(str).unique())

for project in healthy_projects:
    healthy_vals = (
        filtered.loc[
            (filtered["Category"] == "control") &
            (filtered["BioProject Number"].astype(str) == project),
            "Shannon"
        ]
        .dropna()
    )

    if len(healthy_vals) == 0 or len(case_vals_pooled) == 0:
        continue

    u, pval = mannwhitneyu(healthy_vals, case_vals_pooled, alternative="two-sided")
    cliffs_delta = (2 * u) / (len(healthy_vals) * len(case_vals_pooled)) - 1

    results.append({
        "BioProject Number": project,
        "Citation": meta_val.loc[meta_val["BioProject Number"] == project]["Citation"].unique()[0],
        "N_healthy": len(healthy_vals),
        "N_case": len(case_vals_pooled),
        "Cliffs_delta": cliffs_delta,
        "MannWhitney_p": pval,
        "Healthy_median_Shannon": healthy_vals.median(),
        "Case_median_Shannon": case_vals_pooled.median()
    })

healthy_bioproject_vs_cases = pd.DataFrame(results).sort_values("MannWhitney_p")
healthy_bioproject_vs_cases

In [ ]:
# Within-study comparison for cohorts with healthy median NMWI < 0
results = []

paired_projects = set(
    meta_val.loc[meta_val["Category"] == "control", "BioProject Number"]
) & set(
    meta_val.loc[meta_val["Category"] == "case", "BioProject Number"]
)

for project in paired_projects:
    sub = meta_val[meta_val["BioProject Number"] == project]

    healthy = sub.loc[sub["Category"] == "control", "NMWI"].dropna()
    nonhealthy = sub.loc[sub["Category"] == "case", "NMWI"].dropna()

    # Keep only studies where healthy median is below zero
    if healthy.empty or nonhealthy.empty or healthy.median() >= 0:
        continue

    u, p = mannwhitneyu(healthy, nonhealthy, alternative="two-sided")
    delta = 2 * u / (len(healthy) * len(nonhealthy)) - 1

    results.append({
        "Citation": sub["Citation"].iloc[0],
        "BioProject Number": project,
        "N_healthy": len(healthy),
        "N_nonhealthy": len(nonhealthy),
        "Healthy_median": healthy.median(),
        "Nonhealthy_median": nonhealthy.median(),
        "Cliffs_delta": delta,
        "MannWhitney_p": p
    })

within_study_results = pd.DataFrame(results).sort_values("MannWhitney_p")
within_study_results